In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import f1_score, precision_score, recall_score
import os

def find_file(filename, search_path='/kaggle/input'):
    for root, dirs, files in os.walk(search_path):
        if filename in files:
            return os.path.join(root, filename)
    return None

features_path = find_file('elliptic_txs_features.csv')
classes_path = find_file('elliptic_txs_classes.csv')

if not (features_path and classes_path):
    raise FileNotFoundError("Files not found. Please check the dataset.")

def load_data_for_lstm(features_path, classes_path):
    features = pd.read_csv(features_path, header=None)
    classes = pd.read_csv(classes_path)

    mapping = {'unknown': -1, '1': 1, '2': 0, 'illicit': 1, 'licit': 0}
    classes['class'] = classes['class'].map(mapping).fillna(-1).astype(int)

    data = pd.concat([features, classes['class']], axis=1)
    
    labeled_data = data[data['class'] != -1]
    
    train_data = labeled_data[labeled_data.iloc[:, 1] <= 34]
    test_data = labeled_data[labeled_data.iloc[:, 1] > 34]

    X_train = train_data.iloc[:, 1:-1].values
    y_train = train_data.iloc[:, -1].values
    
    X_test = test_data.iloc[:, 1:-1].values
    y_test = test_data.iloc[:, -1].values

    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, y_train, X_test, y_test

def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(units=128, return_sequences=False, input_shape=input_shape))
    model.add(Dropout(0.4))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model

X_train, y_train, X_test, y_test = load_data_for_lstm(features_path, classes_path)

X_train_reshaped = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_reshaped = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

weights_dict = {0: 1.0, 1: 5.0}

model = build_lstm_model((1, X_train.shape[1]))

history = model.fit(
    X_train_reshaped, 
    y_train, 
    epochs=50, 
    batch_size=128, 
    class_weight=weights_dict, 
    verbose=1
)

y_pred_probs = model.predict(X_test_reshaped)

best_f1 = 0
best_thresh = 0.5

for thresh in np.arange(0.1, 0.95, 0.05):
    preds = (y_pred_probs > thresh).astype(int)
    score = f1_score(y_test, preds, pos_label=1, average='binary')
    if score > best_f1:
        best_f1 = score
        best_thresh = thresh

y_pred_final = (y_pred_probs > best_thresh).astype(int)
precision = precision_score(y_test, y_pred_final, pos_label=1)
recall = recall_score(y_test, y_pred_final, pos_label=1)

print("-" * 40)
print(f"FINAL LSTM Results (Optimized)")
print("-" * 40)
print(f"Best Threshold: {best_thresh:.2f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"Illicit F1:     {best_f1:.4f}")
print("-" * 40)

2026-02-10 10:29:32.806560: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770719373.031449      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770719373.110021      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770719373.655430      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770719373.655479      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770719373.655482      55 computation_placer.cc:177] computation placer alr

Epoch 1/50


I0000 00:00:1770719403.677656     125 cuda_dnn.cc:529] Loaded cuDNN version 91002


234/234 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.8376 - loss: 0.7341
Epoch 2/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8903 - loss: 0.4009
Epoch 3/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9216 - loss: 0.3261
Epoch 4/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9315 - loss: 0.2942
Epoch 5/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9430 - loss: 0.2565
Epoch 6/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9495 - loss: 0.2390
Epoch 7/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9456 - loss: 0.2347
Epoch 8/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9574 - loss: 0.2054
Epoch 9/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9563 - loss: 0.1991
Epoch 10/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9576 - loss: 0.2067
Epoch 11/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9592 - loss: 0.1969
Epoch 12/50
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy